In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dropout
)

In [2]:
from google.colab import files

uploaded = files.upload()

Saving movies_genres.csv to movies_genres.csv
Saving movies_overview.csv to movies_overview.csv


In [3]:
movies = pd.read_csv("movies_overview.csv")

genres = pd.read_csv("movies_genres.csv")

In [4]:
movies.head()

,title,overview,genre_ids
0,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]"
1,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]"
2,The Godfather Part II,In the continuing saga of the Corleone crime f...,"[18, 80]"
3,Schindler's List,The true story of how businessman Oskar Schind...,"[18, 36, 10752]"
4,12 Angry Men,The defense and the prosecution have rested an...,[18]


In [5]:
genres.head()

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime


In [6]:
print("Movies Shape:", movies.shape)

print("Genres Shape:", genres.shape)

Movies Shape: (10000, 3)
Genres Shape: (19, 2)


In [7]:
print(movies.columns)

print(genres.columns)

Index(['title', 'overview', 'genre_ids'], dtype='object')
Index(['id', 'name'], dtype='object')


In [8]:
movies.isnull().sum()

,0
title,0
overview,1
genre_ids,0


In [9]:
movies = movies.dropna(
    subset=['overview']
)

print(movies.shape)

(9999, 3)


In [10]:
movies['genre_ids'].head(10)

,genre_ids
0,"[18, 80]"
1,"[18, 80]"
2,"[18, 80]"
3,"[18, 36, 10752]"
4,[18]
5,"[16, 10751, 14]"
6,"[18, 28, 80, 53]"
7,"[35, 18, 10749]"
8,"[14, 18, 80]"
9,"[35, 53, 18]"


In [11]:
import ast

def get_first_genre(x):

    try:
        return ast.literal_eval(x)[0]

    except:
        return np.nan

movies["genre"] = movies["genre_ids"].apply(
    get_first_genre
)

movies.head()

,title,overview,genre_ids,genre
0,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]",18.0
1,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]",18.0
2,The Godfather Part II,In the continuing saga of the Corleone crime f...,"[18, 80]",18.0
3,Schindler's List,The true story of how businessman Oskar Schind...,"[18, 36, 10752]",18.0
4,12 Angry Men,The defense and the prosecution have rested an...,[18],18.0


In [12]:
genre_dict = dict(
    zip(
        genres["id"],
        genres["name"]
    )
)

genre_dict

{28: 'Action',
 12: 'Adventure',
 16: 'Animation',
 35: 'Comedy',
 80: 'Crime',
 99: 'Documentary',
 18: 'Drama',
 10751: 'Family',
 14: 'Fantasy',
 36: 'History',
 27: 'Horror',
 10402: 'Music',
 9648: 'Mystery',
 10749: 'Romance',
 878: 'Science Fiction',
 10770: 'TV Movie',
 53: 'Thriller',
 10752: 'War',
 37: 'Western'}

In [13]:
movies["genre_name"] = movies["genre"].map(
    genre_dict
)

movies.head()

,title,overview,genre_ids,genre,genre_name
0,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]",18.0,Drama
1,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]",18.0,Drama
2,The Godfather Part II,In the continuing saga of the Corleone crime f...,"[18, 80]",18.0,Drama
3,Schindler's List,The true story of how businessman Oskar Schind...,"[18, 36, 10752]",18.0,Drama
4,12 Angry Men,The defense and the prosecution have rested an...,[18],18.0,Drama


In [14]:
movies = movies.dropna(
    subset=[
        "overview",
        "genre_name"
    ]
)

print(movies.shape)

(9996, 5)


In [15]:
movies["genre_name"].value_counts()

,count
genre_name,
Drama,2236
Comedy,1974
Action,1281
Horror,881
Animation,580
Adventure,517
Thriller,513
Crime,454
Romance,319


In [16]:
X = movies["overview"]

y = movies["genre_name"]

In [17]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(np.unique(y))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]


In [18]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)

In [19]:
print(X.iloc[0])

print(X_seq[0])

Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope.
[1812, 7, 2, 4025, 14, 2, 1341, 148, 6, 8, 77, 5, 11, 612, 9516, 4026, 2189, 1, 112, 3, 33, 27, 30, 2, 1, 222, 57, 13, 625, 8, 1, 679, 4, 158, 14, 12, 9517, 3350, 92, 8, 162, 4027, 7, 222, 1, 168, 4, 38, 8059, 20, 2, 90, 3151, 470, 12, 582, 1759, 132, 872, 14, 8, 6999, 5, 1, 917, 6, 407]


In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_pad = pad_sequences(
    X_seq,
    maxlen=100,
    padding='post',
    truncating='post'
)

print(X_pad.shape)

(9996, 100)


In [21]:
X_train, X_test, y_train, y_test = train_test_split(

    X_pad,
    y,

    test_size=0.2,

    random_state=42
)

In [22]:
print(X_train.shape)

print(X_test.shape)

print(y_train.shape)

print(y_test.shape)

(7996, 100)
(2000, 100)
(7996,)
(2000,)


In [23]:
num_classes = len(np.unique(y))

print("Number of Classes:", num_classes)

Number of Classes: 18


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout

model = Sequential()

model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=100
    )
)

model.add(Flatten())

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(
    Dropout(0.3)
)

model.add(
    Dense(
        64,
        activation='relu'
    )
)

model.add(
    Dropout(0.3)
)

model.add(
    Dense(
        num_classes,
        activation='softmax'
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [26]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=32
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.2079 - loss: 2.4685 - val_accuracy: 0.2400 - val_loss: 2.3531
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.2910 - loss: 2.1576 - val_accuracy: 0.3200 - val_loss: 2.2114
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.6232 - loss: 1.1953 - val_accuracy: 0.2856 - val_loss: 2.5305
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.8748 - loss: 0.4141 - val_accuracy: 0.2756 - val_loss: 3.1910
Epoch 5/10
 68/200 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.9407 - loss: 0.2019

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Training vs Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend([
    "Training",
    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend([
    "Training",
    "Validation"
])

plt.show()

In [ ]:
loss, ann_accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy =", ann_accuracy)

In [ ]:
y_pred = model.predict(X_test)

y_pred = np.argmax(
    y_pred,
    axis=1
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(12,8))

sns.heatmap(
    cm,
    annot=False,
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

rnn_model = Sequential()

rnn_model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=100
    )
)

rnn_model.add(
    SimpleRNN(
        64
    )
)

rnn_model.add(
    Dropout(0.3)
)

rnn_model.add(
    Dense(
        64,
        activation='relu'
    )
)

rnn_model.add(
    Dense(
        num_classes,
        activation='softmax'
    )
)

In [ ]:
rnn_model.summary()

In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_rnn = rnn_model.fit(
    X_train,
    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=32
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_rnn.history['accuracy'])
plt.plot(history_rnn.history['val_accuracy'])

plt.title("RNN Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend([
    "Train",
    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_rnn.history['loss'])
plt.plot(history_rnn.history['val_loss'])

plt.title("RNN Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend([
    "Train",
    "Validation"
])

plt.show()

In [ ]:
loss, rnn_accuracy = rnn_model.evaluate(
    X_test,
    y_test
)

print("Accuracy =", rnn_accuracy)

In [ ]:
y_pred_rnn = rnn_model.predict(
    X_test
)

y_pred_rnn = np.argmax(
    y_pred_rnn,
    axis=1
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_rnn
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred_rnn
)

plt.figure(figsize=(12,8))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.show()

In [ ]:
rnn_model.save(
    "movie_genre_rnn.h5"
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

lstm_model = Sequential()

lstm_model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=100
    )
)

lstm_model.add(
    LSTM(
        64
    )
)

lstm_model.add(
    Dropout(0.3)
)

lstm_model.add(
    Dense(
        64,
        activation='relu'
    )
)

lstm_model.add(
    Dense(
        num_classes,
        activation='softmax'
    )
)

In [ ]:
lstm_model.summary()

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_lstm = lstm_model.fit(
    X_train,
    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=32
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_lstm.history['accuracy'])
plt.plot(history_lstm.history['val_accuracy'])

plt.title("LSTM Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend([
    "Train",
    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_lstm.history['loss'])
plt.plot(history_lstm.history['val_loss'])

plt.title("LSTM Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend([
    "Train",
    "Validation"
])

plt.show()

In [ ]:
loss, lstm_accuracy = lstm_model.evaluate(
    X_test,
    y_test
)

print("Accuracy =", lstm_accuracy)

In [ ]:
y_pred_lstm = lstm_model.predict(
    X_test
)

y_pred_lstm = np.argmax(
    y_pred_lstm,
    axis=1
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_lstm
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred_lstm
)

plt.figure(figsize=(12,8))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.title("LSTM Confusion Matrix")

plt.show()

In [ ]:
lstm_model.save(
    "movie_genre_lstm.h5"
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

gru_model = Sequential()

gru_model.add(
    Embedding(
        input_dim=10000,
        output_dim=128,
        input_length=100
    )
)

gru_model.add(
    GRU(
        64
    )
)

gru_model.add(
    Dropout(0.3)
)

gru_model.add(
    Dense(
        64,
        activation='relu'
    )
)

gru_model.add(
    Dense(
        num_classes,
        activation='softmax'
    )
)

In [ ]:
gru_model.summary()

In [ ]:
gru_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_gru = gru_model.fit(
    X_train,
    y_train,

    validation_split=0.2,

    epochs=10,

    batch_size=32
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_gru.history['accuracy'])
plt.plot(history_gru.history['val_accuracy'])

plt.title("GRU Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend([
    "Train",
    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_gru.history['loss'])
plt.plot(history_gru.history['val_loss'])

plt.title("GRU Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend([
    "Train",
    "Validation"
])

plt.show()

In [ ]:
loss, gru_accuracy = gru_model.evaluate(
    X_test,
    y_test
)

print("Accuracy =", gru_accuracy)

In [ ]:
y_pred_gru = gru_model.predict(
    X_test
)

y_pred_gru = np.argmax(
    y_pred_gru,
    axis=1
)
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_gru
    )
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_gru
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred_gru
)

plt.figure(figsize=(12,8))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.title("GRU Confusion Matrix")

plt.show()

In [ ]:
gru_model.save(
    "movie_genre_gru.h5"
)

In [ ]:
results = pd.DataFrame({

    "Model":[
        "ANN",
        "RNN",
        "LSTM",
        "GRU"
    ],

    "Accuracy":[
        ann_accuracy,
        rnn_accuracy,
        lstm_accuracy,
        gru_accuracy
    ]
})

print(results)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x='Model',
    y='Accuracy',
    data=results
)

plt.title("Model Comparison")

plt.show()